In [ ]:
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, concatenate, BatchNormalization, Activation
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2

In [ ]:
# === Paso 1: Definir rutas de datasets ===
print("Paso 1: Definir rutas de los datasets")

pothole_train_img_path = r"C:\Users\marco\AndroidStudioProjects\TFG\TFG-App-IA-IncidenciasViales\code\AI_model\datasets\Segmentacion\pothole600\training\rgb"
pothole_train_mask_path = r"C:\Users\marco\AndroidStudioProjects\TFG\TFG-App-IA-IncidenciasViales\code\AI_model\datasets\Segmentacion\pothole600\training\label"

crack_img_path = r"C:\Users\marco\AndroidStudioProjects\TFG\TFG-App-IA-IncidenciasViales\code\AI_model\datasets\Segmentacion\crack_segmentation_dataset\images"
crack_mask_path = r"C:\Users\marco\AndroidStudioProjects\TFG\TFG-App-IA-IncidenciasViales\code\AI_model\datasets\Segmentacion\crack_segmentation_dataset\masks"

print(f"Archivos en pothole images: {len(os.listdir(pothole_train_img_path))}")
print(f"Archivos en pothole masks: {len(os.listdir(pothole_train_mask_path))}")
print(f"Archivos en crack images: {len(os.listdir(crack_img_path))}")
print(f"Archivos en crack masks: {len(os.listdir(crack_mask_path))}")

In [ ]:
# === Paso 2: Obtener listas completas de imágenes y máscaras ===
print("\nPaso 2: Crear listas con las rutas de imágenes y máscaras")

def get_file_pairs(img_dir, mask_dir):
    imgs = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    masks = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    img_paths = [os.path.join(img_dir, f) for f in imgs]
    mask_paths = [os.path.join(mask_dir, f) for f in masks]
    return img_paths, mask_paths

pothole_imgs, pothole_masks = get_file_pairs(pothole_train_img_path, pothole_train_mask_path)
crack_imgs, crack_masks = get_file_pairs(crack_img_path, crack_mask_path)

print(f"Imágenes pothole600: {len(pothole_imgs)}")
print(f"Máscaras pothole600: {len(pothole_masks)}")
print(f"Imágenes crack: {len(crack_imgs)}")
print(f"Máscaras crack: {len(crack_masks)}")

# Combinar datasets
all_imgs = pothole_imgs + crack_imgs
all_masks = pothole_masks + crack_masks

print(f"Total imágenes combinadas: {len(all_imgs)}")
print(f"Total máscaras combinadas: {len(all_masks)}")

In [ ]:
# === Paso 3: Dividir datos en entrenamiento y validación ===
print("\nPaso 3: Dividir en conjuntos de entrenamiento y validación")

img_train, img_val, mask_train, mask_val = train_test_split(all_imgs, all_masks, test_size=0.2, random_state=42)

print(f"Imágenes entrenamiento: {len(img_train)}")
print(f"Imágenes validación: {len(img_val)}")


In [ ]:
# === Paso 4: Crear generador personalizado para cargar datos en batches ===
print("\nPaso 4: Definir generador para cargar imágenes y máscaras")

class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, img_filenames, mask_filenames, batch_size=16, image_size=(256,256), shuffle=True):
        self.img_filenames = img_filenames
        self.mask_filenames = mask_filenames
        self.batch_size = batch_size
        self.image_size = image_size
        self.shuffle = shuffle
        self.on_epoch_end()
    
    def __len__(self):
        return int(np.floor(len(self.img_filenames) / self.batch_size))
    
    def __getitem__(self, index):
        batch_img_filenames = self.img_filenames[index*self.batch_size:(index+1)*self.batch_size]
        batch_mask_filenames = self.mask_filenames[index*self.batch_size:(index+1)*self.batch_size]
        X, Y = self.__data_generation(batch_img_filenames, batch_mask_filenames)
        return X, Y
    
    def on_epoch_end(self):
        if self.shuffle:
            temp = list(zip(self.img_filenames, self.mask_filenames))
            np.random.shuffle(temp)
            self.img_filenames, self.mask_filenames = zip(*temp)
    
    def __data_generation(self, batch_img_filenames, batch_mask_filenames):
        X = np.empty((self.batch_size, *self.image_size, 3), dtype=np.float32)
        Y = np.empty((self.batch_size, *self.image_size, 1), dtype=np.float32)
        
        for i, (img_path, mask_path) in enumerate(zip(batch_img_filenames, batch_mask_filenames)):
            img = cv2.imread(img_path)
            img = cv2.resize(img, self.image_size)
            img = img / 255.0
            
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, self.image_size)
            mask = mask / 255.0
            mask = np.expand_dims(mask, axis=-1)
            
            X[i,] = img
            Y[i,] = mask
        
        return X, Y

print("Generador definido")

In [ ]:
# === Paso 5: Crear generadores de entrenamiento y validación ===
print("\nPaso 5: Crear generadores de datos")

batch_size = 16
train_gen = DataGenerator(img_train, mask_train, batch_size=batch_size, image_size=(256,256), shuffle=True)
val_gen = DataGenerator(img_val, mask_val, batch_size=batch_size, image_size=(256,256), shuffle=False)

print(f"Tamaño batch: {batch_size}")
print(f"Steps por epoch (train): {len(train_gen)}")
print(f"Steps por epoch (val): {len(val_gen)}")

In [ ]:
# === Paso 6: Definir modelo U-Net con MobileNetV2 como backbone ===
print("\nPaso 6: Definir modelo U-Net con MobileNetV2")

def conv_block(input_tensor, num_filters):
    x = Conv2D(num_filters, 3, padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(num_filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

def decoder_block(input_tensor, concat_tensor, num_filters):
    x = Conv2DTranspose(num_filters, (2,2), strides=(2,2), padding='same')(input_tensor)
    x = concatenate([x, concat_tensor])
    x = conv_block(x, num_filters)
    return x

def build_unet_mobilenetv2(input_shape):
    inputs = Input(shape=input_shape)

    base_model = MobileNetV2(input_tensor=inputs, include_top=False, weights='imagenet')

    for layer in base_model.layers:
        layer.trainable = False  # congelar encoder inicialmente

    skip1 = base_model.get_layer('block_1_expand_relu').output  # 128x128x96
    skip2 = base_model.get_layer('block_3_expand_relu').output  # 64x64x144
    skip3 = base_model.get_layer('block_6_expand_relu').output  # 32x32x192
    skip4 = base_model.get_layer('block_13_expand_relu').output # 16x16x576

    bottleneck = base_model.get_layer('block_16_project').output # 8x8x320

    d1 = decoder_block(bottleneck, skip4, 512)  # 16x16
    d2 = decoder_block(d1, skip3, 256)          # 32x32
    d3 = decoder_block(d2, skip2, 128)          # 64x64
    d4 = decoder_block(d3, skip1, 64)           # 128x128

    d5 = Conv2DTranspose(32, (2,2), strides=(2,2), padding='same')(d4)  # 256x256
    d5 = conv_block(d5, 32)

    outputs = Conv2D(1, (1,1), activation='sigmoid')(d5)

    model = Model(inputs, outputs)
    return model

model = build_unet_mobilenetv2((256,256,3))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("Modelo compilado")
model.summary()

In [ ]:
# === Paso 7: Entrenar modelo ===
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',     
    patience=5,              
    restore_best_weights=True 
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,               
)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# === Paso: Cargar imagen externa ===
print("Paso: Cargar y predecir una imagen externa")

# Ruta de tu imagen
image_path = r"C:\Users\marco\AndroidStudioProjects\TFG\TFG-App-IA-IncidenciasViales\code\AI_model\IMG\cocodrilo.jpg"

# Cargar imagen
image = cv2.imread(image_path)
if image is None:
    raise FileNotFoundError(f"No se pudo cargar la imagen desde {image_path}")

# Convertir de BGR (OpenCV) a RGB
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Redimensionar al tamaño del modelo
image_resized = cv2.resize(image_rgb, (256, 256))
image_input = image_resized.astype(np.float32) / 255.0
image_input = np.expand_dims(image_input, axis=0)  # (1, 256, 256, 3)

# === Paso: Predecir máscara ===
pred_mask = model.predict(image_input)[0]
pred_mask_bin = (pred_mask > 0.5).astype(np.uint8)

# === Paso: Mostrar resultados ===
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Imagen original")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(pred_mask_bin.squeeze(), cmap='gray')
plt.title("Máscara predicha")
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Ruta a tu imagen
image_path = r"C:\Users\marco\AndroidStudioProjects\TFG\TFG-App-IA-IncidenciasViales\code\AI_model\IMG\cocodrilo.jpg"

# Cargar imagen original
img_original = cv2.imread(image_path)
img_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)  # Convertir de BGR a RGB para visualizar bien en matplotlib

# Redimensionar a 224x224
img_resized = cv2.resize(img_original, (224, 224))

# Mostrar ambas imágenes
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img_original)
plt.title(f'Original ({img_original.shape[1]}x{img_original.shape[0]})')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_resized)
plt.title('Redimensionada (224x224)')
plt.axis('off')

plt.tight_layout()
plt.show()